In [1]:
# various import statements
import os
import inspect
import seaborn
import matplotlib
import matplotlib.pyplot as plt
import torch
import scanpy as sc
import pyro

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU is available")
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available")

import sys
import antipode
from antipode.antipode_model import *
import antipode.model_functions
from antipode.model_functions import *
import antipode.model_distributions
from antipode.model_distributions import *
import antipode.model_modules
from antipode.model_modules import *
import antipode.train_utils
from antipode.train_utils import *
import antipode.plotting
from antipode.plotting import *
from antipode.antipode_mixins import AntipodeTrainingMixin, AntipodeSaveLoadMixin


/home/matthew.schmitz/Matthew/utils/miniforge3/envs/antipode/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU is not available


In [2]:
model_tag='1.9.1.8.3_Dev-Free-SecCovLaplace-NoSoftmax-ZINB'#Laplace
discov_key='species'
batch_key='batch_name'

In [3]:
batch_size=32
steps=0
max_steps=100000
num_particles=1
print_every=5000

antipode_model=ANTIPODE.load(os.path.join('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/models',model_tag),prefix='p3_',device=device)
adata=antipode_model.adata_manager.adata
# antipode_model=ANTIPODE.load(os.path.join('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/models',model_tag),prefix='p3_',adata=adata,device=device)


INFO     Generating sequential column names                                                                        


In [ ]:
adata.layers['norm_spliced']=adata.layers['spliced'].copy()
adata.layers['norm_spliced']=scipy.sparse.csr_matrix(1e4*(adata.layers['norm_spliced']/adata.layers['norm_spliced'].sum(1)))

In [ ]:
!nvidia-smi
antipode_model.clear_cuda()
!nvidia-smi

In [ ]:
#plot_loss(antipode_model.losses)

In [ ]:
!nvidia-smi
antipode_model.clear_cuda()
!nvidia-smi

In [ ]:
#plot_loss(antipode_model.losses)
plot_gmm_heatmaps(antipode_model)
plot_d_hists(antipode_model)
plot_batch_embedding_pca(antipode_model)

In [ ]:
prefix=''
pstore=adata.uns['param_store']
n_clusters=antipode_model.level_sizes[-1]
level_edges=[antipode.model_functions.numpy_hardmax(antipode_model.adata_manager.adata.uns[prefix+'param_store']['edges_'+str(i)],axis=-1) for i in range(len(antipode_model.level_sizes)-1)]
levels=antipode_model.tree_convergence_bottom_up.just_propagate(np.eye(antipode_model.level_sizes[-1]),level_edges)
prop_taxon=np.concatenate(levels,axis=-1)

discov_labels=adata.obs[antipode_model.discov_key].cat.categories
latent_labels=[str(x) for x in range(pstore['discov_dc'].shape[1])]
adata.obs['level_2']=adata.obs['level_2'].astype('category')
cluster_index=adata.obs['level_2'].cat.categories.astype(int)#list(range(antipode_model.level_sizes[-1]))#list(range(pstore['locs'].shape[0]))
cluster_labels=list(adata.obs['level_2'].cat.categories)
cluster_label_dict=dict(zip(cluster_index,cluster_labels))
var_labels=adata.var.index

prop_locs=prop_taxon@pstore['locs']
prop_cluster_intercept=prop_taxon@pstore['cluster_intercept']
cluster_params=(prop_locs@pstore['z_decoder_weight'])+prop_cluster_intercept+np.mean(pstore['discov_constitutive_de'],0,keepdims=True)
cluster_params=cluster_params[cluster_index,:]

#Need to propagate multilayer tree to discovs
prop_discov_di = np.einsum('pc,dcg->dpg',prop_taxon,pstore['discov_di'])
prop_discov_dm = np.einsum('pc,dcm->dpm',prop_taxon,pstore['discov_dm'])
discov_cluster_params=np.einsum('dpm,dmg->dpg',prop_locs+prop_discov_dm,pstore['z_decoder_weight']+pstore['discov_dc'])+(prop_cluster_intercept+prop_discov_di+np.expand_dims(pstore['discov_constitutive_de'],1))
discov_cluster_params=discov_cluster_params

In [ ]:
aggr_means=antipode.model_functions.group_aggr_anndata(adata,['species','level_2'],layer='norm_spliced')
log_real_means=np.log(aggr_means[0]+1e-3) # #species,#cluster,#genes array

In [ ]:
real_means=pd.DataFrame(log_real_means.mean(0),columns=adata.var.index,index=aggr_means[1]['level_2'])
real_means=real_means.loc[aggr_means[1]['level_2'],:]

In [ ]:
dev_cons_means=pd.DataFrame(cluster_params,columns=adata.var.index,index=cluster_labels)
dev_cons_means=dev_cons_means.loc[aggr_means[1]['level_2'],:]

In [ ]:
real_dev_mouse_means=pd.DataFrame(log_real_means[2],columns=adata.var.index,index=aggr_means[1]['level_2'])
real_dev_mouse_means=real_dev_mouse_means.loc[aggr_means[1]['level_2'],:]

In [ ]:
fits=[]
for g in (dev_cons_means.columns):
    x=dev_cons_means.loc[:,g]
    y=real_means.loc[:,g]
    fits.append(scipy.stats.stats.spearmanr(x,y).statistic)


In [ ]:
print(np.mean(np.nan_to_num(fits)))
seaborn.histplot(fits)
plt.title('')

In [ ]:
norm = plt.Normalize(-1, 1)

# Create a colormap object
cmap = plt.cm.coolwarm

# Map 'fits' values to the colormap
colors = cmap(np.array(fits))

seaborn.scatterplot(x=real_means.mean(0),y=adata.uns['param_store']['s_inverse_dispersion'],color=colors,s=.3)

In [ ]:
sc.pl.embedding(
    adata,
    basis='X_antipode_MDE',
    color=adata.var.index[np.argsort(fits)][5000:5010],
    cmap='Purples',
    legend_loc='on data'
)

In [ ]:
sc.pl.embedding(
    adata,
    basis='X_antipode_MDE',
    color=adata.var.index[np.argsort(fits)][-12:],
    cmap='Purples',
    legend_loc='on data'
)

In [ ]:
sc.pl.embedding(
    adata,
    basis='X_antipode_MDE',
    color=adata.var.index[np.argsort(fits)][0:12],
    cmap='Purples',
    legend_loc='on data'
)

In [ ]:
# recon_means=antipode.model_functions.group_aggr_anndata(adata,['species','level_2'],layer='reconstructed')
# recon_means=pd.DataFrame(recon_means[0].mean(0),columns=adata.var.index,index=recon_means[1]['level_2'])
# recon_means=recon_means.loc[recon_means[1]['level_2'],:]

In [ ]:
MDE_KEY = "X_antipode_MDE"
#adata.obsm[MDE_KEY] = clip_latent_dimensions(scvi.model.utils.mde(adata.obsm['X_antipode'],init='random'),0.1)
sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=["antipode_cluster"],legend_fontsize=6,legend_fontweight='normal',
    legend_loc='on data',palette=sc.pl.palettes.godsnot_102
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=[x for x in adata.obs.columns if 'level' in x],
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=['q_score',discov_key,batch_key],palette=sc.pl.palettes.godsnot_102,cmap='coolwarm'
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=[x for x in adata.obs.columns if 'psi' in x],
    cmap='coolwarm',
    legend_loc='on data'
)

In [ ]:
MDE_KEY = "X_antipode_MDE"

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color='phase',
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)


In [ ]:

sc.pl.embedding(
    adata,
    basis="X_umap",
    color='phase',
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)

sc.pl.embedding(
    adata,
    basis="X_umap",
    color=[x for x in adata.obs.columns if 'level' in x],
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)


In [ ]:
batch_size=32
steps=0
max_steps=100000
num_particles=1
print_every=5000

antipode_model=ANTIPODE.load(os.path.join('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/models',model_tag),prefix='p3_',device=device)
adata=antipode_model.adata_manager.adata
# antipode_model=ANTIPODE.load(os.path.join('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/models',model_tag),prefix='p3_',adata=adata,device=device)


In [ ]:
adata.layers['norm_spliced']=adata.layers['spliced'].copy()
adata.layers['norm_spliced']=scipy.sparse.csr_matrix(1e4*(adata.layers['norm_spliced']/adata.layers['norm_spliced'].sum(1)))

In [ ]:
!nvidia-smi
antipode_model.clear_cuda()
!nvidia-smi

In [ ]:
#plot_loss(antipode_model.losses)

In [ ]:
!nvidia-smi
antipode_model.clear_cuda()
!nvidia-smi

In [ ]:
#plot_loss(antipode_model.losses)
plot_gmm_heatmaps(antipode_model)
plot_d_hists(antipode_model)
plot_batch_embedding_pca(antipode_model)

In [ ]:
prefix=''
pstore=adata.uns['param_store']
n_clusters=antipode_model.level_sizes[-1]
level_edges=[antipode.model_functions.numpy_hardmax(antipode_model.adata_manager.adata.uns[prefix+'param_store']['edges_'+str(i)],axis=-1) for i in range(len(antipode_model.level_sizes)-1)]
levels=antipode_model.tree_convergence_bottom_up.just_propagate(np.eye(antipode_model.level_sizes[-1]),level_edges)
prop_taxon=np.concatenate(levels,axis=-1)

discov_labels=adata.obs[antipode_model.discov_key].cat.categories
latent_labels=[str(x) for x in range(pstore['discov_dc'].shape[1])]
adata.obs['level_2']=adata.obs['level_2'].astype('category')
cluster_index=adata.obs['level_2'].cat.categories.astype(int)#list(range(antipode_model.level_sizes[-1]))#list(range(pstore['locs'].shape[0]))
cluster_labels=list(adata.obs['level_2'].cat.categories)
cluster_label_dict=dict(zip(cluster_index,cluster_labels))
var_labels=adata.var.index

prop_locs=prop_taxon@pstore['locs']
prop_cluster_intercept=prop_taxon@pstore['cluster_intercept']
cluster_params=(prop_locs@pstore['z_decoder_weight'])+prop_cluster_intercept+np.mean(pstore['discov_constitutive_de'],0,keepdims=True)
cluster_params=cluster_params[cluster_index,:]

#Need to propagate multilayer tree to discovs
prop_discov_di = np.einsum('pc,dcg->dpg',prop_taxon,pstore['discov_di'])
prop_discov_dm = np.einsum('pc,dcm->dpm',prop_taxon,pstore['discov_dm'])
discov_cluster_params=np.einsum('dpm,dmg->dpg',prop_locs+prop_discov_dm,pstore['z_decoder_weight']+pstore['discov_dc'])+(prop_cluster_intercept+prop_discov_di+np.expand_dims(pstore['discov_constitutive_de'],1))
discov_cluster_params=discov_cluster_params

In [ ]:
aggr_means=antipode.model_functions.group_aggr_anndata(adata,['species','level_2'],layer='norm_spliced')
log_real_means=np.log(aggr_means[0]+1e-3) # #species,#cluster,#genes array

In [ ]:
real_means=pd.DataFrame(log_real_means.mean(0),columns=adata.var.index,index=aggr_means[1]['level_2'])
real_means=real_means.loc[aggr_means[1]['level_2'],:]

In [ ]:
dev_cons_means=pd.DataFrame(cluster_params,columns=adata.var.index,index=cluster_labels)
dev_cons_means=dev_cons_means.loc[aggr_means[1]['level_2'],:]

In [ ]:
real_dev_mouse_means=pd.DataFrame(log_real_means[2],columns=adata.var.index,index=aggr_means[1]['level_2'])
real_dev_mouse_means=real_dev_mouse_means.loc[aggr_means[1]['level_2'],:]

In [ ]:
fits=[]
for g in (dev_cons_means.columns):
    x=dev_cons_means.loc[:,g]
    y=real_means.loc[:,g]
    fits.append(scipy.stats.stats.spearmanr(x,y).statistic)


In [ ]:
print(np.mean(np.nan_to_num(fits)))
seaborn.histplot(fits)
plt.title('')

In [ ]:
norm = plt.Normalize(-1, 1)

# Create a colormap object
cmap = plt.cm.coolwarm

# Map 'fits' values to the colormap
colors = cmap(np.array(fits))

seaborn.scatterplot(x=real_means.mean(0),y=adata.uns['param_store']['s_inverse_dispersion'],color=colors,s=.3)

In [ ]:
sc.pl.embedding(
    adata,
    basis='X_antipode_MDE',
    color=adata.var.index[np.argsort(fits)][5000:5010],
    cmap='Purples',
    legend_loc='on data'
)

In [ ]:
sc.pl.embedding(
    adata,
    basis='X_antipode_MDE',
    color=adata.var.index[np.argsort(fits)][-12:],
    cmap='Purples',
    legend_loc='on data'
)

In [ ]:
sc.pl.embedding(
    adata,
    basis='X_antipode_MDE',
    color=adata.var.index[np.argsort(fits)][0:12],
    cmap='Purples',
    legend_loc='on data'
)

In [ ]:
# recon_means=antipode.model_functions.group_aggr_anndata(adata,['species','level_2'],layer='reconstructed')
# recon_means=pd.DataFrame(recon_means[0].mean(0),columns=adata.var.index,index=recon_means[1]['level_2'])
# recon_means=recon_means.loc[recon_means[1]['level_2'],:]

In [ ]:
MDE_KEY = "X_antipode_MDE"
#adata.obsm[MDE_KEY] = clip_latent_dimensions(scvi.model.utils.mde(adata.obsm['X_antipode'],init='random'),0.1)
sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=["antipode_cluster","kmeans"],legend_fontsize=6,legend_fontweight='normal',
    legend_loc='on data',palette=sc.pl.palettes.godsnot_102
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=[x for x in adata.obs.columns if 'level' in x],
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=['q_score',discov_key,batch_key],palette=sc.pl.palettes.godsnot_102,cmap='coolwarm'
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=[x for x in adata.obs.columns if 'psi' in x],
    cmap='coolwarm',
    legend_loc='on data'
)

In [ ]:
MDE_KEY = "X_antipode_MDE"

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color='phase',
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)
